In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import confusion_matrix, classification_report

print("TensorFlow version:", tf.__version__)

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)
print("Testing images:", x_test.shape)
print("Testing labels:", y_test.shape)

In [ ]:
class_names = [
    'airplane',
    'automobile',
    'bird',
    'cat',
    'deer',
    'dog',
    'frog',
    'horse',
    'ship',
    'truck'
]

print(class_names)

In [ ]:
plt.figure(figsize=(12, 8))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

print("Minimum pixel value:", x_train.min())
print("Maximum pixel value:", x_train.max())

In [ ]:
y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test, 10)

print("New training labels shape:", y_train.shape)
print("New testing labels shape:", y_test.shape)

In [ ]:
validation_size = 5000

x_val = x_train[-validation_size:]
y_val = y_train[-validation_size:]

x_train_new = x_train[:-validation_size]
y_train_new = y_train[:-validation_size]

print("Training:", x_train_new.shape)
print("Validation:", x_val.shape)
print("Testing:", x_test.shape)

In [ ]:
def create_ann_model(
    activation='relu',
    optimizer='adam',
    learning_rate=0.001
):

    model = keras.Sequential([

        layers.Input(shape=(32, 32, 3)),

        layers.Flatten(),

        # Hidden Layer 1
        layers.Dense(512, activation=activation),

        # Hidden Layer 2
        layers.Dense(256, activation=activation),

        # Hidden Layer 3
        layers.Dense(128, activation=activation),

        # Output Layer
        layers.Dense(10, activation='softmax')
    ])

    # Optimizer
    if optimizer == 'adam':
        opt = keras.optimizers.Adam(learning_rate=learning_rate)

    elif optimizer == 'sgd':
        opt = keras.optimizers.SGD(learning_rate=learning_rate)

    elif optimizer == 'rmsprop':
        opt = keras.optimizers.RMSprop(learning_rate=learning_rate)

    model.compile(
        optimizer=opt,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
baseline_model = create_ann_model(
    activation='relu',
    optimizer='adam',
    learning_rate=0.001
)

baseline_model.summary()

In [ ]:
start_time = time.time()

baseline_history = baseline_model.fit(
    x_train_new,
    y_train_new,
    validation_data=(x_val, y_val),
    epochs=15,
    batch_size=128,
    verbose=1
)

baseline_training_time = time.time() - start_time

print("Training time:", round(baseline_training_time, 2), "seconds")

In [ ]:
baseline_loss, baseline_accuracy = baseline_model.evaluate(
    x_test,
    y_test,
    verbose=0
)

print("Baseline Test Loss:", baseline_loss)
print("Baseline Test Accuracy:", baseline_accuracy)

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    baseline_history.history['accuracy'],
    label='Training Accuracy'
)

plt.plot(
    baseline_history.history['val_accuracy'],
    label='Validation Accuracy'
)

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy')
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    baseline_history.history['loss'],
    label='Training Loss'
)

plt.plot(
    baseline_history.history['val_loss'],
    label='Validation Loss'
)

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.grid()
plt.show()

In [ ]:
y_pred_prob = baseline_model.predict(x_test)

y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

print("Predictions generated successfully.")

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('CIFAR-10 Confusion Matrix')
plt.show()

In [ ]:
print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names
    )
)

In [ ]:
plt.figure(figsize=(12, 8))

for i in range(20):
    plt.subplot(4, 5, i + 1)

    plt.imshow(x_test[i])

    predicted_class = class_names[y_pred[i]]
    actual_class = class_names[y_true[i]]

    plt.title(
        f"Pred: {predicted_class}\nTrue: {actual_class}"
    )

    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
param_grid = {
    'activation': ['relu', 'tanh'],
    'optimizer': ['adam', 'sgd'],
    'learning_rate': [0.001, 0.01],
    'batch_size': [64, 128]
}

print(param_grid)

In [ ]:
import itertools

grid_results = []

combinations = list(itertools.product(
    param_grid['activation'],
    param_grid['optimizer'],
    param_grid['learning_rate'],
    param_grid['batch_size']
))

print("Total combinations:", len(combinations))

In [ ]:
for activation, optimizer, learning_rate, batch_size in combinations:

    print("\n--------------------------------")
    print("Activation:", activation)
    print("Optimizer:", optimizer)
    print("Learning Rate:", learning_rate)
    print("Batch Size:", batch_size)

    model = create_ann_model(
        activation=activation,
        optimizer=optimizer,
        learning_rate=learning_rate
    )

    start_time = time.time()

    history = model.fit(
        x_train_new,
        y_train_new,
        validation_data=(x_val, y_val),
        epochs=5,
        batch_size=batch_size,
        verbose=0
    )

    training_time = time.time() - start_time

    test_loss, test_accuracy = model.evaluate(
        x_test,
        y_test,
        verbose=0
    )

    best_val_accuracy = max(
        history.history['val_accuracy']
    )

    grid_results.append({
        'activation': activation,
        'optimizer': optimizer,
        'learning_rate': learning_rate,
        'batch_size': batch_size,
        'best_val_accuracy': best_val_accuracy,
        'test_accuracy': test_accuracy,
        'test_loss': test_loss,
        'training_time': training_time,
        'parameters': model.count_params()
    })

In [ ]:
grid_results_df = pd.DataFrame(grid_results)

grid_results_df = grid_results_df.sort_values(
    by='test_accuracy',
    ascending=False
)

grid_results_df

In [ ]:
best_grid = grid_results_df.iloc[0]

print("Best Grid Search Configuration")
print("--------------------------------")

print("Activation:", best_grid['activation'])
print("Optimizer:", best_grid['optimizer'])
print("Learning Rate:", best_grid['learning_rate'])
print("Batch Size:", best_grid['batch_size'])
print("Validation Accuracy:", best_grid['best_val_accuracy'])
print("Test Accuracy:", best_grid['test_accuracy'])
print("Training Time:", best_grid['training_time'])
print("Parameters:", best_grid['parameters'])

In [ ]:
import random

In [ ]:
activation_options = [
    'relu',
    'tanh',
    'sigmoid'
]

optimizer_options = [
    'adam',
    'sgd',
    'rmsprop'
]

learning_rate_options = [
    0.0001,
    0.001,
    0.01
]

batch_size_options = [
    32,
    64,
    128
]

In [ ]:
random_results = []

number_of_trials = 8

for trial in range(number_of_trials):

    activation = random.choice(activation_options)
    optimizer = random.choice(optimizer_options)
    learning_rate = random.choice(learning_rate_options)
    batch_size = random.choice(batch_size_options)

    print("\n================================")
    print("Trial:", trial + 1)
    print("Activation:", activation)
    print("Optimizer:", optimizer)
    print("Learning Rate:", learning_rate)
    print("Batch Size:", batch_size)

    model = create_ann_model(
        activation=activation,
        optimizer=optimizer,
        learning_rate=learning_rate
    )

    start_time = time.time()

    history = model.fit(
        x_train_new,
        y_train_new,
        validation_data=(x_val, y_val),
        epochs=5,
        batch_size=batch_size,
        verbose=0
    )

    training_time = time.time() - start_time

    test_loss, test_accuracy = model.evaluate(
        x_test,
        y_test,
        verbose=0
    )

    best_val_accuracy = max(
        history.history['val_accuracy']
    )

    random_results.append({
        'trial': trial + 1,
        'activation': activation,
        'optimizer': optimizer,
        'learning_rate': learning_rate,
        'batch_size': batch_size,
        'best_val_accuracy': best_val_accuracy,
        'test_accuracy': test_accuracy,
        'test_loss': test_loss,
        'training_time': training_time,
        'parameters': model.count_params()
    })

In [ ]:
random_results_df = pd.DataFrame(random_results)

random_results_df = random_results_df.sort_values(
    by='test_accuracy',
    ascending=False
)

random_results_df

In [ ]:
best_random = random_results_df.iloc[0]

print("Best Random Search Configuration")
print("----------------------------------")

print("Activation:", best_random['activation'])
print("Optimizer:", best_random['optimizer'])
print("Learning Rate:", best_random['learning_rate'])
print("Batch Size:", best_random['batch_size'])
print("Validation Accuracy:", best_random['best_val_accuracy'])
print("Test Accuracy:", best_random['test_accuracy'])
print("Training Time:", best_random['training_time'])
print("Parameters:", best_random['parameters'])

In [ ]:
comparison = pd.DataFrame({
    'Model': [
        'Baseline',
        'Grid Search Best',
        'Random Search Best'
    ],

    'Test Accuracy': [
        baseline_accuracy,
        best_grid['test_accuracy'],
        best_random['test_accuracy']
    ],

    'Test Loss': [
        baseline_loss,
        best_grid['test_loss'],
        best_random['test_loss']
    ],

    'Training Time (sec)': [
        baseline_training_time,
        best_grid['training_time'],
        best_random['training_time']
    ],

    'Parameters': [
        baseline_model.count_params(),
        best_grid['parameters'],
        best_random['parameters']
    ]
})

comparison

In [ ]:
plt.figure(figsize=(10, 6))

plt.bar(
    comparison['Model'],
    comparison['Test Accuracy']
)

plt.ylabel('Test Accuracy')
plt.xlabel('Model')
plt.title('Test Accuracy Comparison')

plt.ylim(0, 1)
plt.xticks(rotation=20)

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.bar(
    comparison['Model'],
    comparison['Training Time (sec)']
)

plt.ylabel('Training Time (seconds)')
plt.xlabel('Model')
plt.title('Training Time Comparison')

plt.xticks(rotation=20)

plt.show()

In [ ]:
if best_grid['test_accuracy'] >= best_random['test_accuracy']:

    final_activation = best_grid['activation']
    final_optimizer = best_grid['optimizer']
    final_learning_rate = best_grid['learning_rate']
    final_batch_size = int(best_grid['batch_size'])

    print("Using Grid Search configuration.")

else:

    final_activation = best_random['activation']
    final_optimizer = best_random['optimizer']
    final_learning_rate = best_random['learning_rate']
    final_batch_size = int(best_random['batch_size'])

    print("Using Random Search configuration.")

print("Activation:", final_activation)
print("Optimizer:", final_optimizer)
print("Learning Rate:", final_learning_rate)
print("Batch Size:", final_batch_size)

In [ ]:
final_model = create_ann_model(
    activation=final_activation,
    optimizer=final_optimizer,
    learning_rate=final_learning_rate
)

final_model.summary()

In [ ]:
start_time = time.time()

final_history = final_model.fit(
    x_train_new,
    y_train_new,
    validation_data=(x_val, y_val),
    epochs=15,
    batch_size=final_batch_size,
    verbose=1
)

final_training_time = time.time() - start_time

print(
    "Final training time:",
    round(final_training_time, 2),
    "seconds"
)

In [ ]:
final_loss, final_accuracy = final_model.evaluate(
    x_test,
    y_test,
    verbose=0
)

print("Final Test Loss:", final_loss)
print("Final Test Accuracy:", final_accuracy)
print("Total Parameters:", final_model.count_params())

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    final_history.history['accuracy'],
    label='Training Accuracy'
)

plt.plot(
    final_history.history['val_accuracy'],
    label='Validation Accuracy'
)

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Final Model - Training vs Validation Accuracy')

plt.legend()
plt.grid()

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    final_history.history['loss'],
    label='Training Loss'
)

plt.plot(
    final_history.history['val_loss'],
    label='Validation Loss'
)

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Final Model - Training vs Validation Loss')

plt.legend()
plt.grid()

plt.show()

In [ ]:
final_predictions = final_model.predict(x_test)

final_y_pred = np.argmax(
    final_predictions,
    axis=1
)

final_y_true = np.argmax(
    y_test,
    axis=1
)

final_cm = confusion_matrix(
    final_y_true,
    final_y_pred
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    final_cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Final Model - Confusion Matrix')

plt.show()

In [ ]:
print(
    classification_report(
        final_y_true,
        final_y_pred,
        target_names=class_names
    )
)

In [ ]:
final_results = pd.DataFrame({
    'Metric': [
        'Test Accuracy',
        'Test Loss',
        'Total Parameters',
        'Training Time (seconds)',
        'Activation',
        'Optimizer',
        'Learning Rate',
        'Batch Size'
    ],

    'Value': [
        final_accuracy,
        final_loss,
        final_model.count_params(),
        round(final_training_time, 2),
        final_activation,
        final_optimizer,
        final_learning_rate,
        final_batch_size
    ]
})

final_results

In [ ]:
final_model.save('CIFAR10_ANN_Final_Model.keras')

print("Model saved successfully.")